# Занятие 2, пара 1. Камера. От света к пикселю

Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai

Сегодня выясним, почему далёкое кажется маленьким, что хранится внутри файла фото и как сохранить копию без метаданных. Нужны NumPy, OpenCV, Matplotlib, Pillow; GPU, телефон и интернет не нужны. Итог: два эксперимента с размером и файл без EXIF. GitHub — по желанию.

In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ExifTags
from course_support.cvtools import imread, show
CAMERA_DIR = DATA_DIR / "camera"
OUTPUT_DIR = PROJECT_DIR / "reports" / "camera"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Разминка: картинка — массив

Прежде чем запускать: каков порядок в shape — ширина или высота? Цветные изображения OpenCV хранит как BGR; show сам переводит их для показа.

In [ ]:
image = imread(CAMERA_DIR / "teaching_photo.jpg")
print(image.shape, image.dtype)
show(image, titles=["Учебный синтетический кадр"])

## Почему далеко — маленькое

Возьмём условный квадрат высотой 1 метр. Его высота в кадре: $h_{px}=f_{px}H/Z$. Здесь H — размер предмета, Z — расстояние, f — фокусное расстояние в пикселях. H и Z должны быть в одинаковых единицах. Это простая модель: предмет стоит прямо перед камерой.

Наш «объектив» не обещает красивое селфи, зато обещает понятную формулу.

In [ ]:
def projected_height(object_height, distance, focal_px=500):
    if distance <= 0:
        raise ValueError("Расстояние должно быть положительным")
    return focal_px * object_height / distance

def draw_object(distance, focal_px=500):
    size = projected_height(1, distance, focal_px)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.add_patch(plt.Rectangle((320-size/2, 240-size/2), size, size, color="steelblue"))
    ax.set(xlim=(0,640), ylim=(480,0), aspect="equal", title=f"Z={distance} м; f={focal_px} px; размер={size:.1f} px")
    plt.show()
    return size

size_demo = draw_object(2)

### ✏️ Ваш ход 1 — отодвинуть предмет

Рассчитайте `near_size` при расстоянии 2 м и `far_size` при 4 м. Размер предмета 1 м, f=500 px. Покажите оба варианта через draw_object. Запишите отношение высот.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('02.1.1', near_size, far_size)

## EXIF: сведения рядом с пикселями

Фото может содержать модель камеры, настройки съёмки и другие метаданные. Может и не содержать — это нормально. Учебный JPEG синтетический, в нём записаны только вымышленные Make/Model/Software. GPS и настоящих сведений о человеке нет.

In [ ]:
with Image.open(CAMERA_DIR / "teaching_photo.jpg") as photo:
    exif_before = {ExifTags.TAGS.get(k, str(k)): str(v) for k,v in photo.getexif().items()}
print(exif_before)

## Сохранение без EXIF

Создадим новое изображение из пикселей и сохраним PNG. Это удаляет EXIF в этой копии, но не скрывает лица, номера и текст внутри самой картинки. Исходник не изменяем.

In [ ]:
with Image.open(CAMERA_DIR / "teaching_photo.jpg") as photo:
    pixels = np.array(photo.convert("RGB"))
clean_path = OUTPUT_DIR / "clean_photo.png"
Image.fromarray(pixels).save(clean_path)
with Image.open(clean_path) as clean:
    print("EXIF после:", dict(clean.getexif()))
    print("Размер:", clean.size)

### ✏️ Ваш ход 2 — проверяем копию

Откройте clean_path через Pillow. Запишите `clean_exif_count` — число EXIF-тегов и `clean_size` — размер. Сравните с исходником. В Markdown укажите, почему отсутствие EXIF не делает фото полностью обезличенным.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('02.1.2', clean_exif_count, clean_size)

## Практический запас: увеличить — не значит вернуть детали, 12–15 минут

Возьмите учебное image. Уменьшите до 80×60, затем увеличьте обратно до исходной ширины и высоты двумя способами: INTER_NEAREST и INTER_LINEAR. Покажите три картинки.

Что выглядит более гладким? Вернулись ли мелкие детали? Где изображение отличается от исходного? Запишите два наблюдения.

Подсказка: размер для cv2.resize задаётся как (ширина, высота), хотя shape начинается с высоты. Опыт необязателен, без дополнительных баллов.

In [ ]:
# ✏️ ваш код здесь

### Ещё 5–8 минут: цель — 100 пикселей

При f=500 px и высоте предмета 1 м подберите расстояние, при котором высота изображения равна 100 px. Сначала оцените, затем используйте projected_height/draw_object. После этого при расстоянии 4 м найдите f для той же высоты. Запишите два разных набора условий, дающих один размер.

## Тема проекта — короткая карточка

Если придумали тему: что классифицируем, какие классы, откуда примеры? Если пока нет — можно предложить «геометрические фигуры» на синтетических картинках (состав набора согласуем) или свою небольшую задачу. Пока это предложение для согласования, не требование срочно собирать фотографии. Запишите один следующий шаг. GitHub и публикация добровольны.

## Для любопытных: матрица камеры

Сначала точка (X,Y,Z) делится на глубину Z, затем масштабируется f и сдвигается к центру кадра (cx,cy). В K эти четыре числа лежат компактно. Умение выводить эту матрицу сейчас не требуется.

In [ ]:
K_demo = np.array([[500.,0,320],[0,500.,240],[0,0,1]])
point = np.array([1.,0,4.])
pixel = K_demo @ point
print("Пиксель:", pixel[:2] / pixel[2])

## Итог

Сохраните тетрадь, clean_photo.png и резервную копию. Можете объяснить изменение размера и отличие пикселей от EXIF — цель достигнута. Во второй паре посмотрим, почему прямые могут выглядеть кривыми.

Подробнее: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html

## Посмотреть или почитать по желанию

- [«Фотоувеличение» / Blow-Up](https://www.criterion.com/films/28694-blow-up) — художественный взгляд на наблюдение и интерпретацию фотографии. Просмотр необязателен.
- Ричард Шелиски, [Computer Vision: Algorithms and Applications](https://szeliski.org/Book/) — справочник для дальнейшего курса. Сейчас достаточно найти раздел о формировании изображения и рассмотреть одну схему камеры; весь учебник читать к следующей паре не нужно.